In [7]:
# In this notebook we will try to solve the prolem of overfitting by using L1 and L2 regularization technique.
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso


In [8]:
df = pd.read_csv('C:\\Users\\compu\\Documents\\GitHub\\GuerraAntonia_DepositoCorso\\corso python\\mercoledì 03-12\\Melbourne_housing.csv')
print("Initial shape:", df.shape)
print(df.nunique())


Initial shape: (34857, 22)
Suburb             351
Address          34009
Rooms               12
Type                 3
Method               9
SellerG            388
Date                78
Distance           215
Postcode           211
Bedroom             15
Bathroom            11
Car                 15
Landsize          1684
BuildingArea       994
YearBuilt          160
CouncilArea         33
Latitude         13402
Longtitude       14524
Regionname           8
Propertycount      342
ParkingArea          8
Price             2871
dtype: int64


C:\Users\compu\AppData\Local\Temp\ipykernel_2628\4243846038.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:\\Users\\compu\\Documents\\GitHub\\GuerraAntonia_DepositoCorso\\corso python\\mercoledì 03-12\\Melbourne_housing.csv')


In [9]:
# so we have several columns with NaN values so we need to handle these columns. We can actually fill some of these column's NaN 
# values just by 0 and some other columns might need some other treatment based on their nature for example price.
# lets first handle the columns where we need to fill only 0.
columns_to_use = ['Suburb', 'Rooms', 'Type', 'Method', 'SellerG', 'Regionname', 'Propertycount', 'Distance', 'CouncilArea', 'Bedroom', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'Price']
df_new = df[columns_to_use].copy()
df_new.head()

,Suburb,Rooms,Type,Method,SellerG,Regionname,Propertycount,Distance,CouncilArea,Bedroom,Bathroom,Car,Landsize,BuildingArea,Price
0,Abbotsford,2,h,SS,Jellis,Northern Metropolitan,4019.0,2.5,Yarra City Council,2.0,1.0,1.0,126.0,inf,NaN
1,Airport West,3,t,PI,Nelson,Western Metropolitan,3464.0,13.5,Moonee Valley City Council,3.0,2.0,1.0,303.0,225,840000.0
2,Albert Park,2,h,S,hockingstuart,Southern Metropolitan,3280.0,3.3,Port Phillip City Council,2.0,1.0,0.0,120.0,82,1275000.0
3,Albert Park,2,h,S,Thomson,Southern Metropolitan,3280.0,3.3,Port Phillip City Council,2.0,1.0,0.0,159.0,inf,1455000.0
4,Alphington,3,h,SN,McGrath,Northern Metropolitan,2211.0,6.4,Darebin City Council,3.0,2.0,1.0,174.0,122,NaN


In [ ]:
# Now lets fill the columns named landsize and building area with mean of the whole respective columns
df_new['Landsize'] = pd.to_numeric(df_new['Landsize'], errors='coerce') # the numbers were in string form so had to convert them to integers.
df_new['BuildingArea'] = pd.to_numeric(df_new['BuildingArea'], errors='coerce')
df_new['Landsize'] = df_new['Landsize'].fillna(df_new.Landsize.mean())
df_new['BuildingArea'] = df_new['BuildingArea'].fillna(df_new.BuildingArea.mean())
df_new = df_new.replace([np.inf, -np.inf], np.nan).dropna()
print("Shape dopo i valori nulli:\n", df_new.isna().sum())



Shape dopo i valori nulli:
 Rooms                                     0
Propertycount                             0
Distance                                  0
Bedroom                                   0
Bathroom                                  0
                                         ..
CouncilArea_Whitehorse City Council       0
CouncilArea_Whittlesea City Council       0
CouncilArea_Wyndham City Council          0
CouncilArea_Yarra City Council            0
CouncilArea_Yarra Ranges Shire Council    0
Length: 642, dtype: int64


Le dummy variables servono a:

-Convertire categorie in numeri

-Evitare interpretazioni errate

-Permettere ai modelli di leggere correttamente variabili non numeriche

In [29]:
# now we are good to go with out cleaned data. Now we are going to make dummy variables for our whole dataset.
df_new = pd.get_dummies(df_new, drop_first=True) # it is a short cut to avoid dummy variable trap it is just dropping the main column whose dummies we have produced. 
df_new

,Rooms,Propertycount,Distance,Bedroom,Bathroom,Car,Landsize,BuildingArea,Price,Suburb_Aberfeldie,...,CouncilArea_Moorabool Shire Council,CouncilArea_Moreland City Council,CouncilArea_Nillumbik Shire Council,CouncilArea_Port Phillip City Council,CouncilArea_Stonnington City Council,CouncilArea_Whitehorse City Council,CouncilArea_Whittlesea City Council,CouncilArea_Wyndham City Council,CouncilArea_Yarra City Council,CouncilArea_Yarra Ranges Shire Council
1,3,3464.0,13.5,3.0,2.0,1.0,303.000000,225.0,840000.0,False,...,False,False,False,False,False,False,False,False,False,False
2,2,3280.0,3.3,2.0,1.0,0.0,120.000000,82.0,1275000.0,False,...,False,False,False,True,False,False,False,False,False,False
5,4,2211.0,6.4,3.0,2.0,4.0,853.000000,263.0,2000000.0,False,...,False,False,False,False,False,False,False,False,False,False
7,3,5301.0,13.8,3.0,2.0,1.0,352.000000,242.0,520000.0,False,...,False,False,False,False,False,False,False,False,False,False
9,5,5132.0,11.1,5.0,3.0,6.0,592.000000,251.0,1085000.0,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34848,2,3873.0,6.4,2.0,1.0,1.0,300.000000,108.0,611500.0,False,...,False,False,False,False,False,False,False,False,False,False
34851,3,11806.0,22.7,3.0,1.0,6.0,569.000000,130.0,627500.0,False,...,False,False,False,False,False,False,False,False,False,False
34852,3,21650.0,12.0,3.0,1.0,1.0,593.598993,105.0,475000.0,False,...,False,False,False,False,False,False,False,False,False,False
34853,4,5833.0,20.6,4.0,2.0,2.0,593.598993,225.0,591000.0,False,...,False,False,False,False,False,False,False,False,False,False


In [33]:
from sklearn.model_selection import train_test_split
x = df_new.drop('Price', axis='columns')
y = df_new.Price
# Now we can jump into our machine learning model and lets first use the train_test_split method

x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=2)
print("Shape of X:", x.shape)
print("Shape of y:", y.shape)
print("Train set X:", x_train.shape)
print("Train set y:", y_train.shape)
print("Test set X:", x_test.shape)
print("Test set y:", y_test.shape)

Shape of X: (10479, 641)
Shape of y: (10479,)
Train set X: (8383, 641)
Train set y: (8383,)
Test set X: (2096, 641)
Test set y: (2096,)


In [35]:
model = LinearRegression()
model.fit(x_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [16]:
model.score(x_test, y_test)
# Our model is much overfit with the training dataset that its accuracy in negative when we provide it with testing dataset. 

0.7238523130029927

In [17]:
model.score(x_train, y_train) # at the same our model is performing very well with respect to the training datset

0.722218936948693

In [18]:
# so we can see that our model is facing the problem of overfitting because on training dataset it scores higher and on the
# testing dataset it score lower. In simple words our model is overfit to the training dataset and underfit to the testing dataset.
# We can solve the problem of overfitting by using L1 0r L2 regularization.  
from sklearn.linear_model import Lasso    # Sklearn's Lass regression is the L1 regularization. 
lasso_model = Lasso()
lasso_model.fit(x_train, y_train)
# the L1 regularization or the Lasso model will add an absolute θ value in the mean squared error

c:\Users\compu\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.300e+14, tolerance: 3.809e+11
  model = cd_fast.enet_coordinate_descent(


,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [19]:
lasso_model.score(x_test, y_test)
# We can see that from -48 percent score to 70 percent score our model is much bette now after using L1 regularization.

0.7240356764797222

In [20]:
lasso_model.score(x_train, y_train)

0.722210577174854

In [21]:
# Now we will use the L2 regularization tehnique
from sklearn.linear_model import Ridge
ridge_model = Ridge(alpha=50, max_iter=100, tol=0.1)
ridge_model.fit(x_train, y_train)

,alpha,50
,fit_intercept,True
,copy_X,True
,max_iter,100
,tol,0.1
,solver,'auto'
,positive,False
,random_state,None


In [22]:
ridge_model.score(x_test,y_test)
# after using L2 regularization our model is also much better but it seems that L1 regularization is slightly better then L2 in this case.

0.7011450723720148

In [23]:
ridge_model.score(x_train,y_train)

0.6853723808022057